# Catalogue search and exact selection

Read the Platform/GDS catalogue, inspect exact source identities and retain selected loader bindings. These cells start no research or execution.

In [ ]:
from acceptance_helpers import connect, check_services

epoch = connect()
await check_services()

## 1. Read published source descriptions

Read constructor details from the same published reference version used for authoring.

In [ ]:
catalog = await epoch.dataset_catalog(source_ids=["cash_flow", "mb_vintage_data"], include_saved=False)
print(catalog.overview)
for source in catalog.sources:
    print(source.documentation)

## 2. Find an instrument and its data product

Use an explicit product category to select the intended dataset.

In [ ]:
from epoch_sdk.discovery_types import DiscoveryQuery
from acceptance_helpers import identities

found = await epoch.search_dataset([DiscoveryQuery(
    id="company", query="Microsoft", mode="discover",
    filters={"kinds": ["asset"], "asset_classes": ["Stocks"], "categories": ["DailyBars"]},
)])
identities(found)

## 3. Resolve a complete current universe

This is separate from ranked name search. The saved reference represents the whole usable cohort; an acquisition gap must remain visible.

In [ ]:
from pydantic import TypeAdapter
from stratifyx.types import ScreenAssetsRequest

cohort = await epoch.screen_assets(TypeAdapter(ScreenAssetsRequest).validate_python({
    "kind": "screen", "screen": {"all_of": [], "any_of": [], "index": ["S&P 500"], "index_membership_mode": "CurrentSnapshot"},
}))
print("Members:", cohort.asset_count)
print("Universe:", cohort.universe_ref)
print("Qualifications:", cohort.qualifications)
assert cohort.universe_ref is not None, "Read the native cohort gaps before using this universe."

## 4. Inspect and retain the chosen source

Read the full record for an explicit candidate, then save its binding. Reopening the selection does not run another search.

In [ ]:
candidate = found.candidates[0]
details = await epoch.catalogue.details(found.session_id, [candidate.id])
print(details.model_dump_json(indent=2))
selection = await epoch.select_sources(found.session_id, [candidate.id])
reopened = await epoch.catalogue.selection(selection.id)
assert reopened == selection
print(selection.model_dump_json(indent=2))

In [ ]:
await epoch.aclose()